# PySpark Advanced Transformations

**Topics:** Type casting, column operations, computed columns, filtering, sorting, limiting

In [1]:
# Initialize Spark Session
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Transformations-part2")
    .master("local[*]")
    .getOrCreate()
)

In [2]:
# Define schema and sample data with data quality issues

emp_schema = """
    emp_id STRING, 
    dept_id STRING, 
    name STRING, 
    age STRING, 
    gender STRING, 
    hire_date STRING,
    salary STRING
"""

emp_data = [
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],
    ["003", "101", "Priya S", "24", "Female", "2024-03-10", "62000"],
    ["001", "101", "Vishnu", "21", "Male", "2025-01-01", "45000"],  # duplicate
    ["004", "103", "Meena R", "", "Female", "2022-11-20", ""],  # missing age & salary
    ["005", "102", "Saravanan", "35", "Male", "", "95000"],
    ["006", "104", "Karthik P", "29", "Male", "2024-09-05", "72000"],
    ["007", "", "Deepika Menon", "26", "Female", "2023-02-28", "68000"],  # missing dept
    ["008", "101", "Mohan Raj", "42", "Male", "2020-05-12", "120000"],
    ["009", "103", "Anjali", "23", "F", "2024-07-19", "58000"],
    ["010", "102", "Ramesh Kumar", "31", "M", "2021-10-01", "85000"],
    ["011", "105", "Swathi", "", "Female", "2025-02-10", "52000"],
    ["002", "102", "Arun Kumar", "28", "Male", "2023-06-15", "78000"],  # duplicate
    ["012", "101", "Vikram Singh", "38", "Male", "2019-08-25", "105000"],
    ["013", "104", "Preethi K", "27", "Female", "", "64000"],
    ["014", "103", "Naveen", "", "Male", "2024-01-15", "null"],  # explicit null
    ["015", "102", "Lavanya", "24", "Female", "2024-04-30", "61000"],
    ["016", "", "Sundar", "45", "M", "2018-03-05", "92000"],
    ["017", "101", "Kavya Sri", "22", "F", "2025-03-01", "48000"],
    ["018", "106", "Abdul Rahman", "33", "Male", "2022-12-12", "88000"],
]

In [3]:
# Create DataFrame
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [4]:
emp.show()

+------+-------+-------------+---+------+----------+------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|
+------+-------+-------------+---+------+----------+------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000|
|   004|    103|      Meena R|   |Female|2022-11-20|      |
|   005|    102|    Saravanan| 35|  Male|          | 95000|
|   006|    104|    Karthik P| 29|  Male|2024-09-05| 72000|
|   007|       |Deepika Menon| 26|Female|2023-02-28| 68000|
|   008|    101|    Mohan Raj| 42|  Male|2020-05-12|120000|
|   009|    103|       Anjali| 23|     F|2024-07-19| 58000|
|   010|    102| Ramesh Kumar| 31|     M|2021-10-01| 85000|
|   011|    105|       Swathi|   |Female|2025-02-10| 52000|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|
|   012|    101| Vikram Singh| 38|  Male

In [5]:
# Type casting: Convert age to INT and salary to DOUBLE

from pyspark.sql.functions import cast, col

emp_casted = emp.select(
    "emp_id", "name", col("age").cast("int"), "gender", col("salary").cast("double")
)

In [6]:
# Verify types changed: age is now INT, salary is DOUBLE
emp_casted.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)


In [7]:
# Empty strings converted to NULL after casting
emp_casted.show()

+------+-------------+----+------+--------+
|emp_id|         name| age|gender|  salary|
+------+-------------+----+------+--------+
|   001|       Vishnu|  21|  Male| 45000.0|
|   002|   Arun Kumar|  28|  Male| 78000.0|
|   003|      Priya S|  24|Female| 62000.0|
|   001|       Vishnu|  21|  Male| 45000.0|
|   004|      Meena R|NULL|Female|    NULL|
|   005|    Saravanan|  35|  Male| 95000.0|
|   006|    Karthik P|  29|  Male| 72000.0|
|   007|Deepika Menon|  26|Female| 68000.0|
|   008|    Mohan Raj|  42|  Male|120000.0|
|   009|       Anjali|  23|     F| 58000.0|
|   010| Ramesh Kumar|  31|     M| 85000.0|
|   011|       Swathi|NULL|Female| 52000.0|
|   002|   Arun Kumar|  28|  Male| 78000.0|
|   012| Vikram Singh|  38|  Male|105000.0|
|   013|    Preethi K|  27|Female| 64000.0|
|   014|       Naveen|NULL|  Male|    NULL|
|   015|      Lavanya|  24|Female| 61000.0|
|   016|       Sundar|  45|     M| 92000.0|
|   017|    Kavya Sri|  22|     F| 48000.0|
|   018| Abdul Rahman|  33|  Mal

In [8]:
# Add computed column: Calculate 20% tax on salary

emp_with_tax = emp_casted.withColumn("tax", col("salary") * 0.2)

In [9]:
# Tax column added (NULL where salary is NULL)
emp_with_tax.show()

+------+-------------+----+------+--------+-------+
|emp_id|         name| age|gender|  salary|    tax|
+------+-------------+----+------+--------+-------+
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|
|   003|      Priya S|  24|Female| 62000.0|12400.0|
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|
|   004|      Meena R|NULL|Female|    NULL|   NULL|
|   005|    Saravanan|  35|  Male| 95000.0|19000.0|
|   006|    Karthik P|  29|  Male| 72000.0|14400.0|
|   007|Deepika Menon|  26|Female| 68000.0|13600.0|
|   008|    Mohan Raj|  42|  Male|120000.0|24000.0|
|   009|       Anjali|  23|     F| 58000.0|11600.0|
|   010| Ramesh Kumar|  31|     M| 85000.0|17000.0|
|   011|       Swathi|NULL|Female| 52000.0|10400.0|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|
|   012| Vikram Singh|  38|  Male|105000.0|21000.0|
|   013|    Preethi K|  27|Female| 64000.0|12800.0|
|   014|       Naveen|NULL|  Male|    NULL|   NULL|
|   015|    

In [10]:
# Add literal column: isPresent flag with value 1

from pyspark.sql.functions import lit

emp_with_flag = emp_with_tax.withColumn("isPresent", lit(1))

In [11]:
# All rows have isPresent = 1
emp_with_flag.show()

+------+-------------+----+------+--------+-------+---------+
|emp_id|         name| age|gender|  salary|    tax|isPresent|
+------+-------------+----+------+--------+-------+---------+
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|        1|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|        1|
|   003|      Priya S|  24|Female| 62000.0|12400.0|        1|
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|        1|
|   004|      Meena R|NULL|Female|    NULL|   NULL|        1|
|   005|    Saravanan|  35|  Male| 95000.0|19000.0|        1|
|   006|    Karthik P|  29|  Male| 72000.0|14400.0|        1|
|   007|Deepika Menon|  26|Female| 68000.0|13600.0|        1|
|   008|    Mohan Raj|  42|  Male|120000.0|24000.0|        1|
|   009|       Anjali|  23|     F| 58000.0|11600.0|        1|
|   010| Ramesh Kumar|  31|     M| 85000.0|17000.0|        1|
|   011|       Swathi|NULL|Female| 52000.0|10400.0|        1|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|        1|
|   012|

In [12]:
# Sort by salary in descending order

emp_sorted = emp_with_flag.orderBy(col("salary").desc())

In [13]:
# Data sorted by salary (highest first, NULLs at bottom)
emp_sorted.show()

+------+-------------+----+------+--------+-------+---------+
|emp_id|         name| age|gender|  salary|    tax|isPresent|
+------+-------------+----+------+--------+-------+---------+
|   008|    Mohan Raj|  42|  Male|120000.0|24000.0|        1|
|   012| Vikram Singh|  38|  Male|105000.0|21000.0|        1|
|   005|    Saravanan|  35|  Male| 95000.0|19000.0|        1|
|   016|       Sundar|  45|     M| 92000.0|18400.0|        1|
|   018| Abdul Rahman|  33|  Male| 88000.0|17600.0|        1|
|   010| Ramesh Kumar|  31|     M| 85000.0|17000.0|        1|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|        1|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|        1|
|   006|    Karthik P|  29|  Male| 72000.0|14400.0|        1|
|   007|Deepika Menon|  26|Female| 68000.0|13600.0|        1|
|   013|    Preethi K|  27|Female| 64000.0|12800.0|        1|
|   003|      Priya S|  24|Female| 62000.0|12400.0|        1|
|   015|      Lavanya|  24|Female| 61000.0|12200.0|        1|
|   009|

In [14]:
# Alternative sorting: Using sort() method

emp_sorted_asc = emp_with_flag.sort(col("salary").asc())

In [15]:
# Sorted by salary ascending (lowest first, NULLs at top)
emp_sorted_asc.show()

+------+-------------+----+------+--------+-------+---------+
|emp_id|         name| age|gender|  salary|    tax|isPresent|
+------+-------------+----+------+--------+-------+---------+
|   004|      Meena R|NULL|Female|    NULL|   NULL|        1|
|   014|       Naveen|NULL|  Male|    NULL|   NULL|        1|
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|        1|
|   001|       Vishnu|  21|  Male| 45000.0| 9000.0|        1|
|   017|    Kavya Sri|  22|     F| 48000.0| 9600.0|        1|
|   011|       Swathi|NULL|Female| 52000.0|10400.0|        1|
|   009|       Anjali|  23|     F| 58000.0|11600.0|        1|
|   015|      Lavanya|  24|Female| 61000.0|12200.0|        1|
|   003|      Priya S|  24|Female| 62000.0|12400.0|        1|
|   013|    Preethi K|  27|Female| 64000.0|12800.0|        1|
|   007|Deepika Menon|  26|Female| 68000.0|13600.0|        1|
|   006|    Karthik P|  29|  Male| 72000.0|14400.0|        1|
|   002|   Arun Kumar|  28|  Male| 78000.0|15600.0|        1|
|   002|

In [16]:
# Filter: Get employees with salary > 75000 (high earners)

emp_filtered = emp_with_flag.filter(col("salary") > 75000)

In [16]:
# Only employees earning more than 75000
emp_filtered.show()

+------+------------+---+------+--------+-------+---------+
|emp_id|        name|age|gender|  salary|    tax|isPresent|
+------+------------+---+------+--------+-------+---------+
|   002|  Arun Kumar| 28|  Male| 78000.0|15600.0|        1|
|   005|   Saravanan| 35|  Male| 95000.0|19000.0|        1|
|   008|   Mohan Raj| 42|  Male|120000.0|24000.0|        1|
|   010|Ramesh Kumar| 31|     M| 85000.0|17000.0|        1|
|   002|  Arun Kumar| 28|  Male| 78000.0|15600.0|        1|
|   012|Vikram Singh| 38|  Male|105000.0|21000.0|        1|
|   016|      Sundar| 45|     M| 92000.0|18400.0|        1|
|   018|Abdul Rahman| 33|  Male| 88000.0|17600.0|        1|
+------+------------+---+------+--------+-------+---------+


In [17]:
# Limit results to top 5 rows

emp_limit = emp_filtered.limit(5)

# Alternative: emp_filtered.show(5)

In [18]:
# First 5 high earners
emp_limit.show()

+------+------------+---+------+--------+-------+---------+
|emp_id|        name|age|gender|  salary|    tax|isPresent|
+------+------------+---+------+--------+-------+---------+
|   002|  Arun Kumar| 28|  Male| 78000.0|15600.0|        1|
|   005|   Saravanan| 35|  Male| 95000.0|19000.0|        1|
|   008|   Mohan Raj| 42|  Male|120000.0|24000.0|        1|
|   010|Ramesh Kumar| 31|     M| 85000.0|17000.0|        1|
|   002|  Arun Kumar| 28|  Male| 78000.0|15600.0|        1|
+------+------------+---+------+--------+-------+---------+


In [19]:
# Add multiple columns at once using withColumns

columns = {"tax": col("salary") * 0.2, "column1": lit(1), "column2": lit("two")}

emp_final = emp.withColumns(columns)

In [20]:
# Three columns added at once: tax (computed), column1 (int), column2 (string)
emp_final.show()

+------+-------+-------------+---+------+----------+------+-------+-------+-------+
|emp_id|dept_id|         name|age|gender| hire_date|salary|    tax|column1|column2|
+------+-------+-------------+---+------+----------+------+-------+-------+-------+
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000| 9000.0|      1|    two|
|   002|    102|   Arun Kumar| 28|  Male|2023-06-15| 78000|15600.0|      1|    two|
|   003|    101|      Priya S| 24|Female|2024-03-10| 62000|12400.0|      1|    two|
|   001|    101|       Vishnu| 21|  Male|2025-01-01| 45000| 9000.0|      1|    two|
|   004|    103|      Meena R|   |Female|2022-11-20|      |   NULL|      1|    two|
|   005|    102|    Saravanan| 35|  Male|          | 95000|19000.0|      1|    two|
|   006|    104|    Karthik P| 29|  Male|2024-09-05| 72000|14400.0|      1|    two|
|   007|       |Deepika Menon| 26|Female|2023-02-28| 68000|13600.0|      1|    two|
|   008|    101|    Mohan Raj| 42|  Male|2020-05-12|120000|24000.0|      1| 